# Тестовое задание: решение

Данные: `Данные для тестового задания.xlsx` (листы «Данные об аудитории», «Данные АБ тестов», «Листеры»).

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

FILE = 'Данные для тестового задания.xlsx'
aud = pd.read_excel(FILE, sheet_name='Данные об аудитории')
ab = pd.read_excel(FILE, sheet_name='Данные АБ тестов')
lst = pd.read_excel(FILE, sheet_name='Листеры')

# в конце листов есть пустые строки - убираем
aud, ab, lst = aud.dropna(how='all'), ab.dropna(how='all'), lst.dropna(how='all')
aud['date'] = pd.to_datetime(aud['date'])
print(aud.shape, ab.shape, lst.shape)
aud.head()

(16814, 3) (2835, 4) (158, 6)


,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


## 1. MAU
Число уникальных пользователей за ноябрь.

In [2]:
mau = aud['user_id'].nunique()
mau

7639

## 2. DAU
Среднее за месяц число уникальных пользователей в день.

In [3]:
dau_by_day = aud.groupby('date')['user_id'].nunique()
print('дней:', len(dau_by_day))
dau = dau_by_day.mean()
round(dau, 1)

дней: 30


np.float64(560.5)

## 3. Retention 1-го дня для пришедших 1 ноября
Новые = первый визит 1 ноября; вернувшиеся = есть визит 2 ноября.

In [4]:
first_visit = aud.groupby('user_id')['date'].min()
cohort = first_visit[first_visit == '2023-11-01'].index
day2_users = set(aud.loc[aud['date'] == '2023-11-02', 'user_id'])
returned = sum(u in day2_users for u in cohort)
print('когорта:', len(cohort), 'вернулись:', returned)
print(f'Retention D1 = {returned / len(cohort):.1%}')

когорта: 623 вернулись: 166
Retention D1 = 26.6%


## 5. Пользовательская конверсия в просмотр объявления
Доля пользователей, у которых за ноябрь хотя бы один просмотр.

In [5]:
views_per_user = aud.groupby('user_id')['view_adverts'].sum()
conv = (views_per_user > 0).mean()
print((views_per_user > 0).sum(), '/', mau, f'= {conv:.1%}')

3538 / 7639 = 46.3%


## 6. Среднее число просмотров на пользователя

In [6]:
avg_views = aud['view_adverts'].sum() / mau
round(avg_views, 2)

np.float64(2.87)

## 7. NPS

In [7]:
promoters, detractors, total = 1200, 500, 2000
nps = promoters / total - detractors / total
f'NPS = {nps:.0%}'

'NPS = 35%'

## 8. A/B-тесты по ARPU
ARPU = выручка / пользователи. Выручка сильно скошена (нули и выбросы), поэтому кроме t-теста Уэлча считаем тест Манна–Уитни и z-тест для доли платящих.

In [8]:
def prop_ztest(x1, n1, x2, n2):
    p = (x1 + x2) / (n1 + n2)
    z = (x2 / n2 - x1 / n1) / np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))

rows = []
for exp, g in ab.groupby('experiment_num'):
    c = g.loc[g['experiment_group'] == 'control', 'revenue']
    t = g.loc[g['experiment_group'] == 'test', 'revenue']
    _, p_t = stats.ttest_ind(t, c, equal_var=False)
    _, p_mw = stats.mannwhitneyu(t, c, alternative='two-sided')
    _, p_conv = prop_ztest((c > 0).sum(), len(c), (t > 0).sum(), len(t))
    rows.append({
        'exp': int(exp), 'n_control': len(c), 'n_test': len(t),
        'ARPU_control': c.mean(), 'ARPU_test': t.mean(), 'lift_%': (t.mean() / c.mean() - 1) * 100,
        'median_control': c.median(), 'median_test': t.median(),
        'payers_control_%': (c > 0).mean() * 100, 'payers_test_%': (t > 0).mean() * 100,
        'p_ttest_welch': p_t, 'p_mannwhitney': p_mw, 'p_payers': p_conv,
    })
ab_res = pd.DataFrame(rows).set_index('exp').round(4)
ab_res

,n_control,n_test,ARPU_control,ARPU_test,lift_%,median_control,median_test,payers_control_%,payers_test_%,p_ttest_welch,p_mannwhitney,p_payers
exp,,,,,,,,,,,,
1,465,480,722.4602,665.7396,-7.8510,104.0,104.0,59.3548,59.7917,0.6890,0.7960,0.8912
2,465,480,704.6538,332.9292,-52.7528,74.0,52.0,59.3548,59.7917,0.0011,0.0085,0.8912
3,465,480,663.2065,998.6688,50.5819,4.0,156.0,58.4946,59.7917,0.0603,0.0010,0.6851


In [9]:
# проверка: пересекаются ли пользователи между группами, и нет ли дисбаланса групп (SRM)
for exp, g in ab.groupby('experiment_num'):
    sizes = g['experiment_group'].value_counts()
    srm_p = stats.chisquare(sizes).pvalue
    print(f'exp {int(exp)}: дубли user_id = {g["user_id"].duplicated().sum()}, размеры {dict(sizes)}, SRM p = {srm_p:.2f}')

exp 1: дубли user_id = 0, размеры {'test': np.int64(480), 'control': np.int64(465)}, SRM p = 0.63
exp 2: дубли user_id = 0, размеры {'test': np.int64(480), 'control': np.int64(465)}, SRM p = 0.63
exp 3: дубли user_id = 0, размеры {'test': np.int64(480), 'control': np.int64(465)}, SRM p = 0.63


**Выводы**
- **Эксп. 1**: ARPU −7,9%, p (t-тест) ≈ 0,69, Манн–Уитни ≈ 0,80. Различий нет, эффекта на выручку нет.
- **Эксп. 2**: ARPU −52,8%, p ≈ 0,001. Значимое ухудшение за счёт снижения суммы платежа (доля платящих та же). Не внедрять.
- **Эксп. 3**: ARPU +50,6%, p (t-тест) ≈ 0,06 (на 5%-ном уровне не значим из-за выбросов), Манн–Уитни ≈ 0,001. Рост идёт за счёт чека, а не числа платящих. Самый перспективный: подтвердить бутстрапом или продлением теста, затем внедрять.

In [10]:
# бутстрап доверительного интервала разницы ARPU для эксперимента 3
rng = np.random.default_rng(42)
g = ab[ab['experiment_num'] == 3]
c = g.loc[g['experiment_group'] == 'control', 'revenue'].to_numpy()
t = g.loc[g['experiment_group'] == 'test', 'revenue'].to_numpy()
diffs = [rng.choice(t, len(t)).mean() - rng.choice(c, len(c)).mean() for _ in range(10_000)]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'разница ARPU = {t.mean() - c.mean():.1f}, 95% ДИ бутстрапа: [{lo:.1f}; {hi:.1f}]')

разница ARPU = 335.5, 95% ДИ бутстрапа: [-7.5; 696.9]


## 9. Средний доход на пользователя (листеры)
Сначала суммируем выручку по пользователю, потом усредняем. Среднее по строкам было бы средним доходом в день, а не на пользователя.

In [11]:
rev_per_user = lst.groupby('user_id')['revenue'].sum()
print('пользователей:', len(rev_per_user))
print('средний доход на пользователя:', round(rev_per_user.mean(), 1))
print('(для сравнения среднее по строкам:', round(lst['revenue'].mean(), 1), ')')

пользователей: 31
средний доход на пользователя: 156.5
(для сравнения среднее по строкам: 30.7 )


## 10. Медиана возраста пользователя

In [12]:
age_per_user = lst.groupby('user_id')['age'].first()
assert (lst.groupby('user_id')['age'].nunique() == 1).all()  # возраст у пользователя не меняется
age_per_user.median()

np.float64(28.0)

## 18. Конверсия в платёж A vs B
z-тест для двух долей и проверка баланса групп (SRM).

In [13]:
nA, xA, nB, xB = 100_047_501, 1003, 100_001_055, 1099
z, p = prop_ztest(xA, nA, xB, nB)
print(f'CR A = {xA / nA:.6%}, CR B = {xB / nB:.6%}, uplift = {(xB / nB) / (xA / nA) - 1:.1%}')
print(f'z = {z:.2f}, p-value = {p:.3f}')
srm = stats.chisquare([nA, nB])
print(f'SRM: chi2 = {srm.statistic:.1f}, p = {srm.pvalue:.4f}')

CR A = 0.001003%, CR B = 0.001099%, uplift = 9.6%
z = 2.10, p-value = 0.035
SRM: chi2 = 10.8, p = 0.0010


**Вывод**: формально разница значима (p ≈ 0,035), но SRM-проверка показывает несбалансированный сплит (p ≈ 0,001), а абсолютный эффект ничтожен (~96 платежей на 100 млн посетителей). Нужно проверить разбиение трафика, повторить тест и оценить влияние на выручку, и только потом раскатывать B.